# Surya 2 — OCR multilingue, malgache officiellement supporté

Surya est le seul modèle de cette série dont la documentation liste le malgache (`mg`, 95,0 % sur le benchmark
interne de 91 langues). Depuis la version 2, la reconnaissance passe par un serveur d'inférence que Surya lance
lui-même au premier appel : **vLLM** (Docker + NVIDIA Container Toolkit) sur GPU NVIDIA, sinon **llama.cpp** —
installez au préalable le binaire `llama-server` (https://github.com/ggml-org/llama.cpp/releases) et mettez-le dans le `PATH`.

Licence des poids : https://github.com/datalab-to/surya/blob/master/MODEL_LICENSE

| | |
|---|---|
| Modèle | [datalab-to/surya](https://github.com/datalab-to/surya) |
| Licence | code Apache-2.0 ; poids OpenRAIL-M modifiée (gratuit < 5 M$ de revenus/financement) |
| Taille | ≈ 0,65 B paramètres |
| Matériel | GPU NVIDIA (vLLM via Docker) ou CPU / Apple Silicon (`llama-server` de llama.cpp) |
| Sortie | blocs en HTML avec polygone, boîte et confiance |
| PDF | pages rastérisées avec `pypdfium2` (fonction commune `load_document`) |

Aucune clé d'API ni token HuggingFace n'est nécessaire : les poids publics sont téléchargés au premier lancement
(dans `~/.cache/huggingface`). Le notebook se lance depuis le dossier `hf_ocr_notebooks/` (il importe `ocr_common.py`).

In [ ]:
%pip install -q surya-ocr pypdfium2 pillow pandas rapidfuzz datasets matplotlib

## 1. Jeux de données

Toutes les sources écrivent dans `data/<source>/` une image par échantillon et, à côté, sa vérité terrain `.txt` :

| Source | Contenu |
|---|---|
| `synthetic/` | 6 images + 1 PDF (français, anglais, tableau, petite police, rotation, bruit) |
| `malagasy/` | 5 images de phrases malgaches intégrées (sans réseau) |
| `hf_Lo-Renz-O_malagasy-sentence/` | phrases malgaches réelles du Hub, rendues en images (bruit / inclinaison) |
| `hf_Teklia_IAM-line/` | vraies images de lignes manuscrites anglaises (IAM) |
| `custom/` | vos documents (image ou PDF, + `nom.txt` optionnel pour mesurer le CER) |

Les datasets du Hub sont téléchargés une seule fois (paquet `datasets`) ; sans réseau, ils sont ignorés.

In [ ]:
import time
from pathlib import Path

from IPython.display import display

import ocr_common as oc

oc.make_test_set()
oc.make_malagasy_set()
for loader in (oc.load_hf_text_dataset, oc.load_hf_image_dataset):
    try:
        loader()
    except Exception as e:  # pas de réseau / pas de paquet datasets
        print(f"{loader.__name__} ignoré : {type(e).__name__}: {e}")

FILES = oc.list_inputs()
print(f"{len(FILES)} fichiers :", ", ".join(f"{f.parent.name}/{f.name}" for f in FILES))
oc.show_dataset([f for f in FILES if f.suffix != ".pdf"])

In [ ]:
import os
import shutil

# Pour réutiliser un serveur déjà lancé :
# os.environ["SURYA_INFERENCE_BACKEND"] = "vllm"            # ou "llamacpp"
# os.environ["SURYA_INFERENCE_URL"] = "http://localhost:8000/v1"
DEVICE = os.environ.get("SURYA_INFERENCE_BACKEND", "auto")
if DEVICE != "vllm" and not shutil.which("llama-server") and not shutil.which("nvidia-smi"):
    print("⚠ ni GPU NVIDIA ni llama-server trouvés : installez llama.cpp avant de charger le modèle")
print("Matériel :", oc.hardware(), "| DEVICE =", DEVICE)

## 2. Chargement du modèle

In [ ]:
t0 = time.perf_counter()
from surya.inference import SuryaInferenceManager
from surya.recognition import RecognitionPredictor

manager = SuryaInferenceManager()          # lance (ou rejoint) le serveur d'inférence
recognition = RecognitionPredictor(manager)
LOAD_S = time.perf_counter() - t0
print(f"Modèle chargé en {LOAD_S:.1f}s sur {DEVICE}")

## 3. Fonction d'extraction


Chaque page renvoie des `blocks` (HTML + polygone). Les blocs sont triés par ordre de lecture, le HTML est converti
en texte brut. Pour un découpage par mise en page, passez `layouts` (voir `surya.layout.LayoutPredictor`).

In [ ]:
def _get(obj, key):
    return obj[key] if isinstance(obj, dict) else getattr(obj, key)


def predict(image):
    page = recognition([image])[0]
    blocks = sorted((b for b in _get(page, "blocks") if not _get(b, "skipped")),
                    key=lambda b: _get(b, "reading_order"))
    text = "\n".join(oc.html_to_text(_get(b, "html")) for b in blocks)
    return text, [_get(b, "polygon") for b in blocks]


def extract(image):
    return predict(image)[0]

## 4. Validation visuelle

Pour chaque image : à gauche la page avec les zones détectées (en rouge), à droite la vérité terrain et le texte lu,
avec le CER dans le titre. Commencez par une image malgache, puis parcourez tout le jeu de test.

In [ ]:
img = oc.load_document(oc.MALAGASY_DIR / "mg_vaovao.png")[0]
t = time.perf_counter()
text, boxes = predict(img)
print(f"({time.perf_counter() - t:.1f}s)\n{text}")
oc.show_result(img, text, boxes, ref=oc.ground_truth(oc.MALAGASY_DIR / "mg_vaovao.png"), title="mg_vaovao.png")

In [ ]:
# toutes les sources sauf les PDF ; augmentez max_files pour tout voir
oc.visual_check(predict, [f for f in FILES if f.suffix != ".pdf"], max_files=12)

### PDF multi-pages

In [ ]:
pdf_path = oc.SYNTH_DIR / "rapport_2_pages.pdf"
for i, page in enumerate(oc.load_document(pdf_path), 1):
    t = time.perf_counter()
    text = extract(page)
    print(f"--- page {i} ({time.perf_counter() - t:.1f}s)\n{text}\n")

## 5. Benchmark sur tout le jeu de test

Temps par page et précision (CER / WER) sur tous les fichiers. Le premier appel inclut un éventuel temps de compilation/« chauffe ».
Les textes extraits sont enregistrés dans `results/<modèle>/` et les mesures dans `results/<modèle>.csv`
(utilisées par le notebook `00_comparaison.ipynb`).

In [ ]:
MODEL_NAME = "Surya-2"
df = oc.run_benchmark(extract, MODEL_NAME, FILES)
oc.save_meta(MODEL_NAME, repo="datalab-to/surya", load_s=LOAD_S, device=str(DEVICE))
display(oc.summarize(df).to_frame(MODEL_NAME))

In [ ]:
display(df[[c for c in ("file", "type", "pages", "s_per_page", "cer", "wer", "error") if c in df]].round(3))
mg = df[df.file.str.startswith("mg_") | df.file.str.fullmatch(r"\d{3}\.png")]
if "cer" in mg and len(mg):
    print(f"CER moyen sur les échantillons malgaches et HF : {mg.cer.mean():.1%}")
oc.show_comparison(df, n=3)

## 6. Utiliser le modèle sur vos propres fichiers

Remplacez le chemin ci-dessous par une image ou un PDF ; le texte est enregistré à côté du fichier (`.ocr.txt`).

In [ ]:
def extract_file(path, dpi=200, max_pages=None):
    path = Path(path)
    pages = oc.load_document(path, dpi=dpi, max_pages=max_pages)
    text = "\n\n".join(extract(p) for p in pages)
    path.with_suffix(".ocr.txt").write_text(text, encoding="utf-8")
    return text


# print(extract_file("data/custom/mon_document.pdf"))